# 3D Surface Explorer

Animate a rotating 3D surface `f(x, y)` with full depth-correct rendering.

---

In [ ]:
import matplotlib
matplotlib.rcParams['animation.embed_limit'] = 100

import ipywidgets as w
from IPython.display import display, HTML, clear_output
import mathplt.animations
from mathplt.animations.graph3d import Graph3DAnimator
from mathplt.core.animator import AnimationConfig
from mathplt.jupyter.widgets import AnimationWidget
from mathplt.core.equation_parser import EquationParser

display(HTML("""
<style>
  .mathplt-panel { background:#1a1a2e; border-radius:10px; padding:16px; margin-bottom:12px; }
  .mathplt-label { color:#a0aec0; font-size:12px; font-weight:600; letter-spacing:.05em;
                   text-transform:uppercase; margin-bottom:4px; }
  .mathplt-title { color:#e2e8f0; font-size:20px; font-weight:700; margin-bottom:4px; }
  .mathplt-sub   { color:#718096; font-size:13px; margin-bottom:12px; }
</style>
<div class='mathplt-panel'>
  <div class='mathplt-title'>3D Surface Explorer</div>
  <div class='mathplt-sub'>Type any f(x,y) equation, tune the controls, then hit <b>Render Animation</b>.</div>
</div>
"""))

In [ ]:
# ── Preset equations ──────────────────────────────────────────────────────────
PRESETS = {
    "Ripple":           "sin(sqrt(x**2 + y**2))",
    "Gaussian wave":    "exp(-0.1*(x**2 + y**2)) * cos(x + y)",
    "Saddle wave":      "sin(x) * cos(y)",
    "Ricker wavelet":   "x * exp(-x**2 - y**2)",
    "Peaks":            "3*(1-x)**2*exp(-x**2-(y+1)**2) - 10*(x/5-x**3-y**5)*exp(-x**2-y**2) - (1/3)*exp(-(x+1)**2-y**2)",
    "Mexican hat":      "(1 - 2*(x**2+y**2)) * exp(-(x**2+y**2))",
    "Hyperbolic":       "x**2 / 4 - y**2 / 4",
    "Twisted ribbon":   "sin(x) * y",
}

CMAPS = ["viridis", "plasma", "inferno", "magma", "cividis",
         "coolwarm", "RdBu", "turbo", "twilight", "hsv"]

# ── Widgets ───────────────────────────────────────────────────────────────────
style       = {"description_width": "140px"}
layout_full = w.Layout(width="100%")
layout_half = w.Layout(width="48%")

preset_btn_style = dict(
    button_style="",
    layout=w.Layout(width="auto", margin="2px"),
)

eq_input = w.Text(
    value=PRESETS["Ripple"],
    placeholder="e.g. sin(sqrt(x**2 + y**2))",
    description="f(x, y) =",
    style=style,
    layout=w.Layout(width="580px"),
)
eq_status = w.HTML(value='<span style="color:#68d391">✓ Valid</span>')

preset_buttons = [
    w.Button(description=name, **preset_btn_style)
    for name in PRESETS
]

x_min   = w.FloatSlider(value=-5, min=-10, max=0,  step=0.5, description="x min",      style=style, layout=layout_half)
x_max   = w.FloatSlider(value=5,  min=0,   max=10, step=0.5, description="x max",      style=style, layout=layout_half)
y_min   = w.FloatSlider(value=-5, min=-10, max=0,  step=0.5, description="y min",      style=style, layout=layout_half)
y_max   = w.FloatSlider(value=5,  min=0,   max=10, step=0.5, description="y max",      style=style, layout=layout_half)
res_sl  = w.IntSlider(  value=50, min=20,  max=100,step=5,   description="Resolution", style=style, layout=layout_half)
fps_sl  = w.IntSlider(  value=20, min=8,   max=30, step=2,   description="FPS",        style=style, layout=layout_half)
dur_sl  = w.FloatSlider(value=8,  min=3,   max=20, step=1,   description="Duration (s)",style=style,layout=layout_half)
spd_sl  = w.FloatSlider(value=1.5,min=0.3, max=5,  step=0.2, description="Spin speed", style=style, layout=layout_half)
elev_sl = w.FloatSlider(value=28, min=5,   max=80, step=1,   description="Elevation",  style=style, layout=layout_half)
bob_sl  = w.FloatSlider(value=8,  min=0,   max=30, step=1,   description="Elev. bob ±",style=style, layout=layout_half)
cmap_dd = w.Dropdown(   options=CMAPS, value="viridis",       description="Color map",  style=style, layout=layout_half)
dark_cb = w.Checkbox(   value=True, description="Dark mode",  style=style, layout=layout_half)

render_btn = w.Button(
    description=" Render Animation",
    button_style="success",
    icon="play",
    layout=w.Layout(width="200px", height="38px"),
)
output = w.Output()

# ── Live validation ───────────────────────────────────────────────────────────
_parser = EquationParser()

def _validate(change=None):
    err = _parser.validate(eq_input.value, variables={"x", "y", "t"})
    if err:
        eq_status.value = f'<span style="color:#fc8181">✗ {err}</span>'
        render_btn.disabled = True
    else:
        eq_status.value = '<span style="color:#68d391">✓ Valid</span>'
        render_btn.disabled = False

eq_input.observe(_validate, names="value")

def _set_preset(name):
    def _handler(btn):
        eq_input.value = PRESETS[name]
    return _handler

for btn, name in zip(preset_buttons, PRESETS):
    btn.on_click(_set_preset(name))

# ── Render ────────────────────────────────────────────────────────────────────
def _render(btn):
    with output:
        clear_output(wait=True)
        print("Rendering… (this may take a moment for high resolution)")

        import matplotlib.pyplot as plt
        plt.close("all")

        config = AnimationConfig(
            fps=fps_sl.value,
            duration_seconds=dur_sl.value,
            figsize=(11, 7),
            dpi=90,
            dark_mode=dark_cb.value,
        )
        animator = Graph3DAnimator(
            config,
            equation=eq_input.value,
            x_range=(x_min.value, x_max.value),
            y_range=(y_min.value, y_max.value),
            resolution=res_sl.value,
            cmap=cmap_dd.value,
            azim_per_frame=spd_sl.value,
            elev_mean=elev_sl.value,
            elev_amplitude=bob_sl.value,
        )
        clear_output(wait=True)
        AnimationWidget(animator).display()

render_btn.on_click(_render)

# ── Layout ────────────────────────────────────────────────────────────────────
display(
    w.VBox([
        # Equation row
        w.HTML("<b style='color:#a0aec0;font-size:12px;text-transform:uppercase;letter-spacing:.05em'>Equation</b>"),
        w.HBox([eq_input, eq_status]),
        w.HTML("<span style='color:#718096;font-size:12px'>Presets:</span>"),
        w.HBox(preset_buttons, layout=w.Layout(flex_flow="row wrap")),

        w.HTML("<hr style='border-color:#2d3748;margin:8px 0'>"),

        # Domain
        w.HTML("<b style='color:#a0aec0;font-size:12px;text-transform:uppercase;letter-spacing:.05em'>Domain</b>"),
        w.HBox([x_min, x_max]),
        w.HBox([y_min, y_max]),

        w.HTML("<hr style='border-color:#2d3748;margin:8px 0'>"),

        # Render settings
        w.HTML("<b style='color:#a0aec0;font-size:12px;text-transform:uppercase;letter-spacing:.05em'>Render</b>"),
        w.HBox([res_sl, fps_sl]),
        w.HBox([dur_sl, spd_sl]),

        w.HTML("<hr style='border-color:#2d3748;margin:8px 0'>"),

        # View settings
        w.HTML("<b style='color:#a0aec0;font-size:12px;text-transform:uppercase;letter-spacing:.05em'>View</b>"),
        w.HBox([elev_sl, bob_sl]),
        w.HBox([cmap_dd, dark_cb]),

        w.HTML("<div style='height:8px'></div>"),
        render_btn,
        output,
    ], layout=w.Layout(padding="8px"))
)